### #Simple LCEL (for understanding the flow)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.1:latest", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment classifier. Respond with only: Positive, Negative, or Neutral."),
    ("human", "Classify this sentiment: {text}")
])

chain = prompt | llm | StrOutputParser() ## - see how and where it is used.
print(chain.invoke({"text": "I love this product!"}))

Positive


### **with retriever*

In [2]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# -------------------
# 1. LLM
# -------------------
llm = ChatOllama(model="llama3.1:latest", temperature=0.2)

# -------------------
# 2. Documents
# -------------------
texts = [
    "I love this product. It works great.",
    "This is the worst thing I have bought.",
    "The product is okay, not bad."
]


In [3]:

# -------------------
# 3. Vector DB
# -------------------
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = FAISS.from_texts(texts, embeddings)

retriever = vectorstore.as_retriever()


In [4]:

# -------------------
# 4. Prompt
# -------------------
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment classifier. Respond only with Positive, Negative, or Neutral."),
    ("human", "Context: {context}\nText: {text}")
])

# -------------------
# 5. Chain
# -------------------
chain = prompt | llm | StrOutputParser()

# -------------------
# 6. Retrieve context manually
# -------------------
query = "I absolutely love this!"

docs = retriever.invoke(query)

context = "\n".join([d.page_content for d in docs])

# -------------------
# 7. Run chain
# -------------------
result = chain.invoke({
    "context": context,
    "text": query
})

print(result)

Positive


### **with retiever and runnable*

In [6]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# -----------------------
# 1. LLM
# -----------------------
llm = ChatOllama(model="llama3.1:latest", temperature=0.2)

# -----------------------
# 2. Sample data
# -----------------------
texts = [
    "I love this product. It is amazing.",
    "This is the worst service I have ever used.",
    "The product is okay, nothing special."
]

# -----------------------
# 3. Embeddings + FAISS
# -----------------------
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = FAISS.from_texts(texts, embedding=embeddings)

retriever = vectorstore.as_retriever()

# -----------------------
# 4. Prompt
# -----------------------
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment classifier. Respond only with Positive, Negative, or Neutral."),
    ("human", "Context: {context}\nText: {question}")
])

# -----------------------
# 5. LCEL Chain
# -----------------------
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# -----------------------
# 6. Run
# -----------------------
print(chain.invoke("It's absolutely crap"))

Negative




---

### *What happens internally**

```
User query
   │
   ▼
Retriever (FAISS similarity search)
   │
   ▼
Context documents
   │
   ▼
Prompt template
   │
   ▼
LLM (Ollama)
   │
   ▼
StrOutputParser
   │
   ▼
Final answer
```

---

